# Suite XTY — The Cpp Types dialect

`Expressions.Cpp.Types` models C++'s type-ids (ISO/IEC 14882, [dcl.name]), a grammar of their own, as mbse-programs has
them: a type-id, its specifiers and its abstract declarator, C++'s own helpers, which a lambda's declarator holds too,
and an enumeration's body, its enumerators and the directives and comments among them, as data. C++'s terms hold its
terms wherever a type stands (a cast's, a named cast's, `new`'s, a conversion function's name's, a trailing return
type, an enumeration's fixed type, a type parameter's default) and where an expression or a type may (`sizeof(int)`),
C++ reading them. Its kinds are validated, bind nothing, cover C++'s three editions, print as mbse-programs prints
them, and are refused by evaluation by name; a cast reads its type from it.

In [ ]:
from mbse.Expressions import Framework
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.C import Expressions as CFamily, Preprocessor as CPreprocessor, Types as CTypes
from mbse.Expressions.Cpp import Expressions as CppFamily, Types as CppTypes
from mbse.Expressions.SystemV import AMD64
from mbse.Expressions.C.Expressions import Expressions as C
from mbse.Expressions.C.Preprocessor import Expressions as PP
from mbse.Expressions.Cpp.Expressions import Evaluators as XV, Expressions as CPP, Text as XText
from mbse.Expressions.Cpp.Types import Expressions as XT, Text as XTText
from mbse.Expressions.Framework import Domains as D, Standards as FS, Symbolics as SY, Terms as F
from mbse.Schemas.Framework import Builders, JSON, Validators
from support import raises

store = Builders.Store()  # the C family's dialects, Cpp Types' install before C++'s: the installs run in any order
for install in [Framework.install, ISOIEC11404.install, IEEE754.install, CppTypes.install, CFamily.install, CppFamily.install,
                CTypes.install, CPreprocessor.install, AMD64.install]:
    store.transaction(install)
for define in [CFamily.define, CppFamily.define, AMD64.define, CppTypes.define]:  # C++'s types and the ABI's, its editions
    store.transaction(define)
AMD64_ = store.Instances.Expressions.SystemV.AMD64  # the ABI a holder names, whose types C++ leaves to the implementation


def term(spec, dialect=XT.DIALECT, abi=AMD64_):
    """The term value `spec` writes, the root of `dialect`'s holder, naming `abi`, written in a transaction."""
    with store.transaction():
        path = dialect.path.split(".")
        holder = getattr(getattr(store.Expressions, path[0]), path[1]).Expression
        return F.term(holder(lambda e: e.root(dialect.spec(spec)).abi([abi])))


def validate(spec, bound=(), dialect=XT.DIALECT):
    return dialect.validate(term(spec, dialect), bound)


text = lambda spec, dialect=XT.DIALECT: XTText.ToText(term(spec, dialect))  # noqa: E731
union = lambda name: [b.name for b in store.registered(name).branches]  # noqa: E731: a union's branches
schemas = lambda name: {b.name: b.schema for b in store.registered(name).branches}  # noqa: E731
spelled = lambda word: lambda i: i.spelling(word)  # noqa: E731: an identifier, of its spelling
named = lambda word: lambda n: n.identifier(spelled(word))  # noqa: E731: a name that is an identifier
keyword = lambda word: lambda s: s.primitivetypespecifier(lambda p: p.keyword(word))  # noqa: E731
type_name = lambda *specifiers, declarator=None: lambda t: t.typeid(  # noqa: E731: a type-id, of its specifiers
    lambda i: i.specifiers(list(specifiers)) if declarator is None else i.specifiers(list(specifiers)).declarator(declarator))
int_ = type_name(keyword("int"))
x = CPP.identifier("x")
enumerator = lambda word, value=None: lambda u: u.enumerator(  # noqa: E731: `word = value`
    lambda r: r.name(spelled(word)) if value is None else r.name(spelled(word)).value(value))
enumeration = lambda items, key="enum class", base=None: type_name(lambda s: s.enumspecifier(  # noqa: E731: `enum class E { items }`
    lambda e: (e.key(key).name(named("E")) if base is None else e.key(key).name(named("E")).base(base)).body(True).enumerators(items)))


<a id="XTY-01"></a>
## XTY-01 · Its union and holder: its kinds, a type-id and an enumerator, invalid syntax, and the preprocessor's directives and comment; a holder refining the framework's, naming its ABI, round-tripped through JSON

In [ ]:
assert XT.TAGS == ("typeid", "enumerator") and list(XT.DIALECT.kinds()) == [*XT.TAGS, "error", "missing"] and XT.HELPERS == ()
assert union(XT.DIALECT.union) == [*XT.TAGS, "error", "missing", *PP.DIRECTIVES, "comment"]
assert XT.DIALECT.name() == "Cpp Types" and XT.DIALECT.tag == "cpp_types" and "cpp_types" in union("Expressions.Framework.Term")
kinds, held = schemas(XT.DIALECT.union), schemas(CPP.DIALECT.union)
assert held["typeid"] is kinds["typeid"] is store.registered("Expressions.Cpp.Types.TypeId")  # one schema, in each union
assert kinds["typeid"] is not schemas("Expressions.C.Types.Term")["typeid"]  # C++'s own, holding C++'s helpers
assert XT.DIALECT.reads() == (CPP.DIALECT, PP.DIALECT) and CPP.DIALECT.kind_named("typeid") is XT.DIALECT.kinds()["typeid"]
Holder = store.registered("Expressions.Cpp.Types.Expression")
assert Holder.ref and Holder.refines is store.registered("Expressions.Framework.Expression") and F.is_expression(Holder)
assert Holder.properties["root"].schema is store.registered(XT.DIALECT.union) and list(Holder.adjacencies) == ["abi"]
with store.transaction():
    holder = store.Expressions.Cpp.Types.Expression(lambda e: e.root(XT.DIALECT.spec(enumeration([enumerator("A")]))).abi([AMD64_]))
assert Validators.Validate(store)(Holder, holder) == [] and XT.DIALECT.validate(holder) == []
assert F.same(JSON.FromJSON(store)(Holder, JSON.ToJSON(store)(Holder, holder)).root, holder.root)
with raises(Exception, match="'binaryexpression' is not a member of a union value"):  # its root holds its own terms
    with store.transaction():
        store.Expressions.Cpp.Types.Expression(lambda e: e.root(XT.DIALECT.spec(CPP.binary("+", x, 1))))


<a id="XTY-02"></a>
## XTY-02 · Its kinds and positions: a type-id's parts C++'s helpers, read through reads; an enumeration's body its terms; every position of a type in C++ holding its Term

In [ ]:
props = lambda path, name: list(store.registered(f"Expressions.{path}.{name}").properties)  # noqa: E731
assert props("Cpp.Types", "TypeId") == ["specifiers", "declarator"] and props("Cpp.Types", "Enumerator") == ["name", "attributes", "value"]
assert props("Cpp.Expressions", "EnumSpecifier") == ["key", "attributes", "name", "base", "body", "enumerators", "trailing_comma"]
prop = lambda path, name, member: store.registered(f"Expressions.{path}.{name}").properties[member].schema  # noqa: E731
assert prop("Cpp.Types", "TypeId", "specifiers").element is store.registered("Expressions.Cpp.Expressions.Specifier")
assert prop("Cpp.Types", "TypeId", "declarator") is store.registered("Expressions.Cpp.Expressions.Declarator")
assert prop("Cpp.Types", "Enumerator", "value") is store.registered(CPP.DIALECT.union)  # C++'s terms
assert prop("Cpp.Types", "Enumerator", "name") is store.registered("Expressions.Cpp.Expressions.Identifier")
assert prop("Cpp.Expressions", "EnumSpecifier", "enumerators").element is store.registered(XT.DIALECT.union)
Types = store.registered(XT.DIALECT.union)
for name, member in [("CastExpression", "type"), ("NamedCastExpression", "type"), ("NewExpression", "type"),
                     ("ConversionName", "type"), ("LambdaDeclarator", "trailing_return"), ("FunctionDeclarator", "trailing_return"),
                     ("EnumSpecifier", "base"), ("TypeParameter", "default")]:
    assert prop("Cpp.Expressions", name, member) is Types, name
with raises(LookupError, match="no schema registered as 'Expressions.Cpp.Expressions.Type'"):  # moved, no alias
    store.registered("Expressions.Cpp.Expressions.Type")
with raises(LookupError, match="no schema registered as 'Expressions.Cpp.Expressions.TypeId'"):
    store.registered("Expressions.Cpp.Expressions.TypeId")


<a id="XTY-03"></a>
## XTY-03 · Its checks: an enumerator needs a name, an identifier's spelling, an enumeration's enumerators within its body, and an enumeration's items of C++ alone

In [ ]:
assert validate(lambda u: u.enumerator(lambda r: r)) == ["an enumerator needs a name"]
assert validate(enumerator("a b")) == ["'a b' is not an identifier"]
assert validate(type_name(lambda s: s.enumspecifier(lambda e: e.key("enum").enumerators([enumerator("A")])))) == [
    "an EnumSpecifier with enumerators has a body"]
assert validate(enumerator("A", CPP.literal(1, "wb"))) == ["value: a literal's suffix cannot be 'wb'"]  # C++'s suffixes
c_item = CTypes.Expressions.DIALECT.spec(lambda u: u.enumerator(lambda r: r.name(spelled("B"))))
directive = XT.DIALECT.spec(lambda u: u.ifdefdirective(lambda d: d.name(spelled("B_")).items([c_item])))
assert validate(enumeration([directive])) == ["specifiers 0 argument 0: argument 0: a C Types enumerator is not a Cpp Types term"]
c_condition = XT.DIALECT.spec(lambda u: u.ifdirective(lambda d: d.condition(C.binary(">", C.identifier("N"), 1))))
assert validate(enumeration([c_condition])) == ["specifiers 0 argument 0: condition: a C binaryexpression is not a Cpp Types term"]


<a id="XTY-04"></a>
## XTY-04 · What it binds: nothing; a lambda's parameters within its trailing return type, a C++ binder's own

In [ ]:
assert not any(F.is_binder(b.schema) for b in store.registered(XT.DIALECT.union).branches)
assert SY.free(term(enumeration([enumerator("A", x)])), XT.DIALECT) == {"x"}
decltype = lambda word: type_name(lambda s: s.decltypespecifier(lambda d: d.expression(CPP.identifier(word))))  # noqa: E731
lambda_ = lambda u: u.lambdaexpression(lambda r: r.declarator(lambda d: d.parameters([  # noqa: E731: [](int a) -> decltype(a) {}
    lambda p: p.parameterdeclaration(lambda q: q.specifiers([keyword("int")]).declarator(lambda e: e.iddeclarator(
        lambda i: i.name(named("a")))))]).trailing_return(decltype("a"))))
assert validate(lambda_, (), CPP.DIALECT) == [] and XText.ToText(term(lambda_, CPP.DIALECT)) == "[](int a) -> decltype(a) {}"
assert SY.free(term(lambda_, CPP.DIALECT), CPP.DIALECT) == set()
assert validate(lambda u: u.lambdaexpression(lambda r: r.declarator(lambda d: d.trailing_return(decltype("a")))), (), CPP.DIALECT) == [
    "declarator trailing_return: specifiers 0 expression: idexpression 'a' is not bound"]


<a id="XTY-05"></a>
## XTY-05 · Within C++'s terms: a type wherever one stands, a Cpp Types term C++ reads, validates, infers, evaluates and prints; written by C++'s writers

In [ ]:
uint8 = type_name(lambda s: s.namedtypespecifier(lambda n: n.name(named("uint8_t"))))
cast = term(CPP.cast(uint8, CPP.binary("+", x, 1)), CPP.DIALECT)
assert [F.dialect_of(a) for a in F.form(cast).arguments] == [XT.DIALECT.union, CPP.DIALECT.union]
assert CPP.DIALECT.validate(cast, {"x"}) == [] and XText.ToText(cast) == "(uint8_t)(x + 1)" and XV.OfAny(cast, {"x": 255}) == 0
assert CPP.DIALECT.infer(cast, {"x": D.Anything}).name() == "uint8_t" and F.same(term(CPP.DIALECT.make(F.form(cast)), CPP.DIALECT), cast)
for spec, source in [
    (lambda u: u.namedcastexpression(lambda n: n.operator("static_cast").type(int_).operand(x)), "static_cast<int>(x)"),
    (lambda u: u.newexpression(lambda n: n.type(int_)), "new int"),
    (lambda u: u.sizeofexpression(lambda s: s.operand(int_)), "sizeof(int)"),
    (lambda u: u.idexpression(lambda r: r.name(lambda n: n.conversionname(lambda c: c.type(int_)))), "operator int"),
    (lambda u: u.lambdaexpression(lambda r: r.template_parameters([lambda t: t.typeparameter(lambda p: p.key("typename").name(
        spelled("T")).default(int_))])), "[]<typename T = int> {}"),
    (lambda u: u.sizeofexpression(lambda s: s.operand(enumeration([enumerator("A")], "enum", int_))), "sizeof(enum E : int {\n    A\n})"),
]:
    assert CPP.DIALECT.validate(term(spec, CPP.DIALECT), {"x"}) == [] and XText.ToText(term(spec, CPP.DIALECT)) == source, source
assert CPP.DIALECT.validate(term(lambda u: u.newexpression(lambda n: n), CPP.DIALECT)) == ["a newexpression needs a type"]


<a id="XTY-06"></a>
## XTY-06 · Invalid syntax: error and missing in its Term, validated, printed as written, refused by name

In [ ]:
assert validate(lambda u: u.error(lambda r: r.text("int int"))) == ["invalid syntax 'int int'"] and text(lambda u: u.error(lambda r: r.text("a b"))) == "a b"
assert validate(lambda u: u.missing(lambda r: r.kind("type"))) == ["missing type"]
typed = term(lambda u: u.namedcastexpression(lambda n: n.operator("static_cast").type(lambda t: t.missing(lambda m: m.kind("type"))).operand(x)), CPP.DIALECT)
assert CPP.DIALECT.validate(typed, {"x"}) == ["type: missing type"] and XText.ToText(typed) == "static_cast<>(x)"
with raises(ValueError, match="Cpp Types error has no text to print"):
    text(lambda u: u.error(lambda r: r))


<a id="XTY-07"></a>
## XTY-07 · Its editions: C++'s three, each by its designation and common name, each with both its kinds

In [ ]:
types = store.Instances.Expressions.Cpp.Types
assert [FS.tags(s) for s in FS.standards(types).values()] == [
    ("ISO/IEC 14882:2017", "C++17"), ("ISO/IEC 14882:2020", "C++20"), ("ISO/IEC 14882:2024", "C++23")]
assert FS.properties(FS.find(types, "C++23"))["revision"] == ("year", 2024) and XT.SINCE == {}
assert all([s.name for s in FS.kinds(e)] == ["Expressions.Cpp.Types.TypeId", "Expressions.Cpp.Types.Enumerator"]
           for e in FS.standards(types).values())
assert store.transaction(CppTypes.define)[0]["Cpp20"] is FS.find(types, "C++20")  # run again, it writes nothing


<a id="XTY-08"></a>
## XTY-08 · Printing: each type-id as mbse-programs' printer writes it, an enumeration's body on lines of its own

In [ ]:
reference = lambda d: d.referencedeclarator(lambda r: r)  # noqa: E731
for spec, source in [
    (int_, "int"), (type_name(lambda s: s.cvqualifier(lambda q: q.keyword("const")), lambda s: s.namedtypespecifier(
        lambda n: n.name(lambda m: m.qualifiedname(lambda q: q.qualifiers([named("std")]).name(named("string"))))), declarator=reference),
     "const std::string&"),
    (enumeration([enumerator("A", CPP.literal(1)), enumerator("B")]), "enum class E {\n    A = 1,\n    B\n}"),
    (enumeration([enumerator("A"), lambda u: u.comment(lambda c: c.text(" B")), XT.DIALECT.spec(lambda u: u.otherdirective(
        lambda d: d.directive("pragma").text("region"))), enumerator("C")]),
     "enum class E {\n    A,\n    // B\n#pragma region\n    C\n}"),
    (enumerator("A", CPP.literal(1)), "A = 1"),
]:
    assert text(spec) == source, (source, text(spec))


<a id="XTY-09"></a>
## XTY-09 · Evaluation: none (Q11); each of its kinds is refused by name by its family's evaluator; a cast reads its type rather than evaluate it

In [ ]:
for spec, tag in [(int_, "typeid"), (enumerator("A"), "enumerator")]:
    with raises(NotImplementedError, match=f"Cpp Types {tag} is not evaluated"):
        XV.OfAny(term(spec), {})
assert XV.OfAny(term(CPP.cast(type_name(keyword("signed"), keyword("char")), 200), CPP.DIALECT)) == -56
with raises(NotImplementedError, match="Cpp namedcastexpression is not evaluated"):
    XV.OfAny(term(lambda u: u.namedcastexpression(lambda n: n.operator("static_cast").type(int_).operand(CPP.literal(1))), CPP.DIALECT))
